# Introduction

In this chapter, we'll deal with files. Files can be used to store any kind of data: images, video, audio, text, documents, fonts, etc. Generally, an extension is used to indicate what kind of data is stored:

- `.jpg`, `.gif` and `.png` are popular image formats.
- `.mp3`, `.m4a`, `.ogg`, `.wav`. `.mid` are typical audio formats.
- `.pdf`, `.docx`, `.html` are used for documents.
- and so on.

## Binary vs Text Files

We distinguish between two kinds of files: binary files and text files. This terminology, even though it's standard, can be a bit confusing: *all* files ultimately contain a long sequence of `0` and `1`s, so in that sense, *all* files are binary. So, let's clarify the exact difference between these two types of file.

- A *text file* stores data in human-readable form. All bytes are to be interpreted as characters (ASCII/Unicode). Examples of human-readable file formats are `html`, `md`, `csv`, `xml`, `json` and `yaml`.
- A *binary file* does not try to be human-readable and can therefore store data in much more compact form. Most file formats are binary: `.jpg`, `.gif`, `.png`, `.mp3`, `docx`, etc.

To actually see the difference, use Visual Studio Code (or any other text editor) to open an `.html` file and a `.jpg`.

### `EXAMPLE`

Consider numbers: in a text file, a number will be written in decimal form, each digit written out as a separate byte. For example, the number `10000` would take up five bytes, as we need five digits to represent it.

In a binary file, we would wonder how many bytes we actually need to represent 10000. The answer is two, as two bytes allow us to represent integers up to 65,535. We could even go further and think in terms of bits: we really only need 14 bits.

In reality it's a little more complicated, but we don't want to fill pages on that topic here. We have to leave something to the other courses :-)

# Reading Files

In this section we discuss how we can read data from text files.

## Opening Files

Files can be read from and written to. That is really their purpose.

However, multiple programs could try to access the same file at the same time. This would lead to chaos: the two programs would overwrite each other's data, and the file would become corrupt, i.e., the data inside it makes no sense. To prevent this, the operating system demands that you *open* a file first. You can then interact with the file. After you're done, you *close* the file.

The OS will make sure no two programs have the same file open at the same time. If you try to open a file that's already in use by another program, you will get an error. It's also important that you close a file, hereby giving others a chance to use the file.

As always, reality is a bit more complex. For example, multiple programs reading from the same file at the same time is perfectly safe. However, once one program needs to write it, it needs exclusive access. This is why when opening a file, you need to mention whether you intend to read or write to it. Based on that, the OS can decide whether to grant you access or not.

Opening and closing a file in Python can be done using

```python .noeval
file = open("my-file.txt")
# interact with file
file.close()
```

This approach is a bit risky however: what if something bad happens while interacting with the file? The `file.close()` statement would then be skipped, making it impossible for other programs to access the file. For this reason, Python offers a special construct:

```python .noeval
with open("my-file.txt") as file:
    # interact with file
```

Here, at the end of the `with` block, the file will automatically be closed. No matter what happens while interacting with the file, it is guaranteed that it will be closed at the end. In other words, you should always rely on `with` when dealing with files.

### `INFO`

If you write code that opens a file, then crashes before closing it again, there's no problem: the OS will clean up after you.

However, a script can recover from crashes (see exceptions). In this case, the `close` operation will be skipped yet your program keeps running. As long as it runs, the file will remain open, making it inaccessible to others. This can become a big problem in the case of long running programs.

## Encoding

There are [many ways](https://en.wikipedia.org/wiki/Character_encoding) texts can be encoded: [ASCII](https://en.wikipedia.org/wiki/ASCII), [UTF-8](https://en.wikipedia.org/wiki/UTF-8), [EBCDIC](https://en.wikipedia.org/wiki/EBCDIC) and so on. These days most files are encoded using UTF-8. It is strongly recommended that you make the encoding explicit when opening the file. This is done as follows:

```python .noeval
with open("my-file.txt", encoding='utf-8') as file:
    # interact with file
```

### `INFO`

The `encoding=` parameter is known as a keyword argument. Suffice it to say that Python allows you to explicitly mention the parameter name, for the sake of clarity. Keyword arguments have a few more advantages, but we won't discuss them in detail right now.

## Reading from Files

Say you open a file using

```python .noeval
with open("my-file.txt", encoding='utf-8') as file:
    # interact with file
```

Inside the `with` block, the variable `file` is set to a special object representing the opened file. Note that `file` is just an identifier: you can pick any name you want.

You can invoke methods on it, just like you could with strings, using the syntax `file.method_name(arguments)`. We discuss some of these methods, specifically those that let you read from the file.

- `file.read()` reads the entire contents of the file and returns it as a *single string*. You probably shouldn't use this approach when the file is large.
- `file.readlines()` reads the entire contents of the file and returns it as an *array of strings*, where each string correspond to a line.
- `file.readline()` reads the next line in the file and returns it as string. If no more lines are left in the file, an empty string is returned.

### `IMPORTANT`

An opened file keeps track of a "current stream position", i.e., it remembers to what point you have read from the file. Initially this position is set to the beginning of the file. `file.readline()` advances this position to the beginning of the next line, so that the next time you invoke `file.readline()` it knows where to look for the next line. `file.read()` and `file.readlines()` both read the entire contents in one go, therefore the position is moved to the end of the file.

### `IMPORTANT`

Whenever you read lines (i.e., using `readline` or `readlines`), know that each string will contain the newline character `\n`. In other words, a line `abc` in the file will be represented by `"abc\n"`. When you open a text file with your text editor of choice, you will not see any of these newline characters `\n`, as they are hidden for our convenience. For our programming languages like python however, they are quite important as we need to explicitly mark we're at the end of the line in our string and we need to move to the next line. In this way, we can create a single string containing three lines:

In [ ]:
my_string = "This is the first line.\nThis is the second line.\nThis is the third line."
print(my_string)

Text files are implicitly built up like this, so we need to take these into account when we read or write our files.

### `EXAMPLE`

Say we have a file `input.txt` with the following contents:

```text
First line
Second line
Third line
Fourth line
Fifth line
```

In [ ]:
# We open it the "careless" way (= not using with) so that
# we can work step by step for the sake of this example
file = open('input.txt', encoding='utf-8')

print(file.readline())
# "First line\n"

print(file.readline())
# "Second line\n"

# Also starts reading from the current stream position
print(file.readlines())
# ["Third line\n", "Fourth line\n", "Fifth line"]

# Current stream position is at end of file, no more lines left
print(file.readline())
# ""

# Close the file at the end
file.close()

In [ ]:
# Normally, we would read our file the following way:
file_content = []
with open('input.txt', encoding='utf-8') as file:
  file_content = file.readlines()
print(file_content)
# ["First line\n", "Second line\n", "Third line\n", "Fourth line\n", "Fifth line"]

# Writing Files

All data a Python program generates disappear as soon as it ends. If you want your program to "remember" data from the previous time it was run, you have to store this data somewhere. Typically this is done using files.

## Opening a File with Write Access

As discussed earlier, files need to be opened prior to interacting with them. By default, we only get read access. In order to get write access, we have to request this explicitly:

```python .noeval
with open(filename, 'w', encoding='utf-8') as file:
    # Interact with file
```

The `'w'` tells the OS we intend to write. Note that opening a file in mode `w` will empty that file: all data that was in it is lost. This can be a useful feature if you just want to clear a file; just open it with write access and close it again.

### `INFO`

The [possible modes](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files) for opening a file are

| Mode | Description |
| :---: | ----------- |
| `'r'`  | Read only |
| `'w'`  | Write only (empties file first!) |
| `'r+'` | Reading and writing |
| `'a'` | Appending (writing at end) |

If you find yourself in a situation where you need to work with binary files instead of text files, you can add a `'b'` to the mode you're using. For example:

| Mode | Description |
| :---: | ----------- |
| `'rb'`  | Read only for a binary file |
| `'wb'`  | Write only, write as binary instead of text (empties file)|
| `'ab'` | Appending to a binary file|

In this course we will focus on text files, so you should mainly remember the first table for file opening modes.

## Writing to a File

The following methods can be used to write to a file:

- `file.write(string)` writes the given `string` to the file. The current stream position moves with it: you can call `write` multiple times and each `string` will be added to the end of the file.
- `file.writelines(strings)` writes all `strings` to the file.

Note that neither of these methods add newlines.

### `EXAMPLE`

In [ ]:
with open('output.txt', 'w', encoding='utf-8') as file:
    file.writelines(['a', 'b', 'c'])

The file \`output.txt' now contains

```text
abc
```

Note how, even though the method is called `writelines`, all strings are written to the same line. In order to put the strings on separate lines, you have to add the newlines yourself:

In [ ]:
with open('output.txt', 'w', encoding='utf-8') as file:
    file.writelines(['a\n', 'b\n', 'c\n'])

## Exercises

### Exercise 50.1: Guest list checker

In the file [guest_list.txt](concept-exercises/guest_list.txt) you can find a list of names of people who have been invited to a highly exclusive event.
Write a simple program which asks for a name of a guest and which then checks whether the guest is on the guest list.

Some example executions:
```none
What is the name of the guest? > Billie Eillish
Billie Eillish is not on the guest list!

What is the name of the guest? > Elliot Page
Elliot Page is on the guest list!
```

Implement this in the file [exercise_50_1_guest_list.py](concept-exercises/exercise_50_1_guest_list.py).
Run the following cell to verify that your implementation is correct:

In [ ]:
# code to run the tests
!python3 -m pytest -q --tb=short concept-exercises/.tests/test_exercise_50_1_guest_list.py

> NOTE: If you get the the error `No such file or directory: 'guest_list.txt'`, when running your program, make sure to use the right launch configuration, namely `Python Debugger: Current File` (from your `launch.json`). If you use on of the other default options `Run Python File` or `Python Debugger: Debug Python File`, it will run your program from the top level folder, meaning it will not find the file `guest_list.txt`. If you have any questions about this, please ask your lecturer.

### Exercise 50.2: Book writer

Write a simple program which allows you to write a "book".
The application should ask the user for a number of sentences until the user quits by pressing (Q), after which the application writes the given sentences to a file named `book.txt`. Every sentence should be a line.

An example flow would be as follows:
```none
Enter a sentence or (Q) quit: > Here is Edward Bear, coming downstairs now, bump, bump, bump, on the back of his head, behind Christopher Robin.
Enter a sentence or (Q) quit: > It is, as far as he knows, the only way of coming downstairs, but sometimes he feels that there really is another way, if only he could stop bumping for a moment and think of it.
Enter a sentence or (Q) quit: > And then he feels that perhaps there isn't.
Enter a sentence or (Q) quit: > Anyhow, here he is at the bottom, and ready to be introduced to you.
Enter a sentence or (Q) quit: > Winnie-the-Pooh.
Enter a sentence or (Q) quit: > Q


The file book.txt then contains:
Here is Edward Bear, coming downstairs now, bump, bump, bump, on the back of his head, behind Christopher Robin.
It is, as far as he knows, the only way of coming downstairs, but sometimes he feels that there really is another way, if only he could stop bumping for a moment and think of it.
And then he feels that perhaps there isn't.
Anyhow, here he is at the bottom, and ready to be introduced to you.
Winnie-the-Pooh.
```

Implement this in the file [exercise_50_2_book_writer.py](concept-exercises/exercise_50_2_book_writer.py).
Run the following cell to verify that your implementation is correct:

In [ ]:
# code to run the tests
!python3 -m pytest -q --tb=short concept-exercises/.tests/test_exercise_50_2_book_writer.py

### Exercise 50.3: Countries visited

You have decided to keep track of the countries that you've visited in the past, which you store in the file [countries.txt](concept-exercises/countries.txt). You now want to be able to add new countries to the list when you've visited them
Write a simple program which asks for a newly visited country and updates the list by adding the entry to it.
The program should also confirm the addition by printing a message to the user.

Hint: you can achieve this without having to read everything in the countries file

Some example executions:
```none
Which country do you want to add? > Portugal
Portugal has been added to the list.

Which country do you want to add? > Turkey
Turkey has been added to the list
```

Implement this in the file [exercise_50_3_countries_visited.py](concept-exercises/exercise_50_3_countries_visited.py).
Run the following cell to verify that your implementation is correct:

In [ ]:
# code to run the tests
!python3 -m pytest -q --tb=short concept-exercises/.tests/test_exercise_50_3_countries_visited.py

### Exercise 50.4: Remove Empty Lines

Write a program that asks for a filename, reads the file, removes the empty lines and writes the results to a file named "output.txt".
If a line has spaces in it, it does not count as an empty line.

Let's say we have a file input.txt containing the following lines:

```
a


b
c

d
```

then the program should work as follows:
```
Provide a filename for the file which should have it's empty lines removed: > input.txt
```
```
output.txt now contains
a
b
c
d
```

Implement this in the file [exercise_50_4_remove_empty_lines.py](concept-exercises/exercise_50_4_remove_empty_lines.py).
Run the following cell to verify that your implementation is correct:

In [ ]:
# code to run the tests
!python3 -m pytest -q --tb=short concept-exercises/.tests/test_exercise_50_4_remove_empty_lines.py